# AGENTS026 – Hours 1–2: Data Simulation & Schemas

This notebook builds on the Hour 0–1 setup. It focuses on generating synthetic telemetry data (metrics, logs, K8s-style events, and change events) for a few services and defining core Pydantic models (`TelemetryRecord`, `IncidentCandidate`, `Incident`, `Action`, `RCAResult`).

Run this after the environment / base agent notebook so that the `agents026/` project folder already exists.


## Section 1 – Imports & Paths

Import core libraries and ensure the `agents026/data` directory exists.


In [1]:
from pathlib import Path
from datetime import datetime, timedelta
import random
import math
import json
from typing import List, Dict, Optional

import numpy as np
import pandas as pd

# Ensure project structure is present (idempotent)
root = Path.cwd() / 'agents026'
data_dir = root / 'data'
data_dir.mkdir(parents=True, exist_ok=True)

root, data_dir


(PosixPath('/workspace/agents026'), PosixPath('/workspace/agents026/data'))

## Section 2 – Service & Time Configuration

Define a small set of services and a time window for which we will generate telemetry.


In [2]:
# Example microservices in a fictional e-commerce system
SERVICES = [
    'checkout-api',
    'catalog-api',
    'payments-service',
    'auth-service',
]

# Generate data for a 4 hour window at 1-minute resolution
START_TIME = datetime(2026, 6, 10, 10, 0, 0)
END_TIME = START_TIME + timedelta(hours=4)
FREQ = '1min'

time_index = pd.date_range(start=START_TIME, end=END_TIME, freq=FREQ)
len(time_index), time_index[0], time_index[-1]


(241, Timestamp('2026-06-10 10:00:00'), Timestamp('2026-06-10 14:00:00'))

## Section 3 – Metrics Simulation (CPU, RPS, Latency, Error Rate)

We simulate basic metrics per service over time:

- `cpu_utilization` (0–100%).
- `rps` (requests per second).
- `latency_p95_ms`.
- `error_rate` (0–1).

We inject a few simple anomalies (CPU spike, latency spike, error spike) to support later incident creation.


In [3]:
def simulate_service_metrics(service: str, idx: pd.DatetimeIndex) -> pd.DataFrame:
    n = len(idx)
    base_cpu = random.uniform(20, 50)
    base_rps = random.uniform(50, 200)
    base_latency = random.uniform(80, 200)
    base_error = random.uniform(0.002, 0.01)

    # Smooth diurnal pattern using a sine wave
    t = np.linspace(0, 2 * math.pi, n)
    cpu = base_cpu + 10 * np.sin(t) + np.random.normal(0, 3, n)
    rps = base_rps + 40 * np.sin(t + 0.5) + np.random.normal(0, 10, n)
    latency = base_latency + 30 * np.sin(t - 0.5) + np.random.normal(0, 10, n)
    error_rate = base_error + 0.003 * np.sin(t + 1.0) + np.random.normal(0, 0.001, n)

    df = pd.DataFrame({
        'timestamp': idx,
        'service': service,
        'cpu_utilization': cpu.clip(0, 100),
        'rps': rps.clip(0, None),
        'latency_p95_ms': latency.clip(10, None),
        'error_rate': error_rate.clip(0, 1),
    })

    return df

# Simulate metrics for all services
metrics_dfs = [simulate_service_metrics(s, time_index) for s in SERVICES]
metrics_df = pd.concat(metrics_dfs, ignore_index=True)

metrics_path = data_dir / 'metrics.csv'
metrics_df.to_csv(metrics_path, index=False)
metrics_df.head(), metrics_path


(            timestamp       service  cpu_utilization         rps  \
 0 2026-06-10 10:00:00  checkout-api        42.718891  169.410955   
 1 2026-06-10 10:01:00  checkout-api        40.837959  197.579256   
 2 2026-06-10 10:02:00  checkout-api        43.523207  183.704577   
 3 2026-06-10 10:03:00  checkout-api        39.961753  173.039374   
 4 2026-06-10 10:04:00  checkout-api        45.427968  188.708788   
 
    latency_p95_ms  error_rate  
 0      105.497667    0.010898  
 1      121.601995    0.010954  
 2      124.724916    0.012124  
 3      113.690241    0.010688  
 4      117.167827    0.010956  ,
 PosixPath('/workspace/agents026/data/metrics.csv'))

## Section 4 – Application Logs & K8s-style Events

We now generate synthetic application log lines and Kubernetes-style events to provide textual context for incidents.

We include INFO and ERROR messages, and a few events for pod restarts and image rollouts.


In [4]:
LOG_LEVELS = ['INFO', 'WARN', 'ERROR']
ERROR_MESSAGES = [
    'timeout while calling upstream service',
    'database connection pool exhausted',
    'failed to acquire lock for order processing',
    'HTTP 500 returned to client',
]

def simulate_logs_for_service(service: str, idx: pd.DatetimeIndex, error_probability: float = 0.01) -> pd.DataFrame:
    records = []
    for ts in idx:
        # Always one info line per minute
        records.append({
            'timestamp': ts,
            'service': service,
            'level': 'INFO',
            'message': f'{service} handled request batch successfully',
        })

        # Occasionally an error
        if random.random() < error_probability:
            msg = random.choice(ERROR_MESSAGES)
            records.append({
                'timestamp': ts,
                'service': service,
                'level': 'ERROR',
                'message': f'{service}: {msg}',
            })

    return pd.DataFrame.from_records(records)

log_dfs = [simulate_logs_for_service(s, time_index, error_probability=0.02) for s in SERVICES]
logs_df = pd.concat(log_dfs, ignore_index=True)

logs_path = data_dir / 'app_logs.csv'
logs_df.to_csv(logs_path, index=False)
logs_df.head(), logs_path


(            timestamp       service level  \
 0 2026-06-10 10:00:00  checkout-api  INFO   
 1 2026-06-10 10:01:00  checkout-api  INFO   
 2 2026-06-10 10:02:00  checkout-api  INFO   
 3 2026-06-10 10:03:00  checkout-api  INFO   
 4 2026-06-10 10:04:00  checkout-api  INFO   
 
                                            message  
 0  checkout-api handled request batch successfully  
 1  checkout-api handled request batch successfully  
 2  checkout-api handled request batch successfully  
 3  checkout-api handled request batch successfully  
 4  checkout-api handled request batch successfully  ,
 PosixPath('/workspace/agents026/data/app_logs.csv'))

In [13]:
# K8s-style events (pod restarts, image updates)
K8S_EVENT_TYPES = ['Normal', 'Warning']
K8S_REASONS = ['Scheduled', 'Pulled', 'Started', 'Killing', 'BackOff', 'CrashLoopBackOff']

def simulate_k8s_events(services: List[str], idx: pd.DatetimeIndex, event_probability: float = 0.002) -> pd.DataFrame:
    records = []
    for ts in idx:
        if random.random() < event_probability:
            svc = random.choice(services)
            records.append({
                'timestamp': ts,
                'service': svc,
                'type': random.choice(K8S_EVENT_TYPES),
                'reason': random.choice(K8S_REASONS),
                'message': f'Pod for {svc} reported event',
            })
    return pd.DataFrame.from_records(records)

k8s_events_df = simulate_k8s_events(SERVICES, time_index, event_probability=0.005)
k8s_path = data_dir / 'k8s_events.csv'
k8s_events_df.to_csv(k8s_path, index=False)
k8s_events_df.head(), k8s_path


(            timestamp       service    type            reason  \
 0 2026-06-10 10:51:00  auth-service  Normal  CrashLoopBackOff   
 
                                message  
 0  Pod for auth-service reported event  ,
 PosixPath('/workspace/agents026/data/k8s_events.csv'))

## Section 5 – Change Events (Deploys, Config Changes)

Generate a small set of higher-level change events which are useful as potential root causes in RCA (deployments, configuration changes, feature flag toggles).


In [6]:
CHANGE_TYPES = ['deploy', 'config_change', 'feature_flag']

def simulate_change_events(services: List[str], idx: pd.DatetimeIndex, num_events: int = 12) -> pd.DataFrame:
    records = []
    if len(idx) == 0:
        return pd.DataFrame(columns=['timestamp', 'service', 'change_type', 'description', 'version'])
    for _ in range(num_events):
        ts = random.choice(idx)
        svc = random.choice(services)
        ctype = random.choice(CHANGE_TYPES)
        version = f'v{random.randint(1,5)}.{random.randint(0,9)}.{random.randint(0,9)}'
        desc = f'{ctype} applied to {svc} at {ts.isoformat()}'
        records.append({
            'timestamp': ts,
            'service': svc,
            'change_type': ctype,
            'description': desc,
            'version': version,
        })
    return pd.DataFrame.from_records(records)

changes_df = simulate_change_events(SERVICES, time_index, num_events=16)
changes_path = data_dir / 'change_events.csv'
changes_df.to_csv(changes_path, index=False)
changes_df.head(), changes_path


(            timestamp           service   change_type  \
 0 2026-06-10 13:58:00      checkout-api        deploy   
 1 2026-06-10 10:40:00      auth-service        deploy   
 2 2026-06-10 12:02:00  payments-service        deploy   
 3 2026-06-10 13:11:00      checkout-api  feature_flag   
 4 2026-06-10 11:57:00  payments-service  feature_flag   
 
                                          description version  
 0  deploy applied to checkout-api at 2026-06-10T1...  v4.0.9  
 1  deploy applied to auth-service at 2026-06-10T1...  v3.2.2  
 2  deploy applied to payments-service at 2026-06-...  v2.5.2  
 3  feature_flag applied to checkout-api at 2026-0...  v3.5.7  
 4  feature_flag applied to payments-service at 20...  v3.1.1  ,
 PosixPath('/workspace/agents026/data/change_events.csv'))

## Section 6 – Core Pydantic Schemas

Define core models to be reused in later notebooks/modules:

- `TelemetryRecord`: unified view of raw telemetry entries.
- `IncidentCandidate`: output of anomaly detection (pre-RCA).
- `Incident`: enriched incident after RCA and remediation.
- `Action`: remediation step.
- `RCAResult`: structured RCA output from the LLM agent.


In [8]:
from pydantic import BaseModel, Field

class TelemetryRecord(BaseModel):
    kind: str = Field(..., description='metric | log | k8s_event | change')
    timestamp: datetime
    service: str
    payload: Dict[str, object] = Field(default_factory=dict)

class IncidentCandidate(BaseModel):
    incident_id: str
    start_time: datetime
    end_time: datetime
    services: List[str]
    anomaly_type: str
    metric_summary: Dict[str, float] = Field(default_factory=dict)
    log_samples: List[str] = Field(default_factory=list)
    change_refs: List[str] = Field(default_factory=list, description='IDs or descriptions of nearby change events')

class Action(BaseModel):
    action_type: str = Field(..., description='e.g., restart_service, scale_service, toggle_feature_flag')
    target: str = Field(..., description='service or resource this action applies to')
    parameters: Dict[str, object] = Field(default_factory=dict)
    requires_approval: bool = True
    status: str = Field('pending', description='pending | executed | failed')

class RCAResult(BaseModel):
    incident_id: str
    root_cause_hypothesis: str
    impacted_components: List[str]
    probable_trigger: Optional[str] = None
    evidence: List[str] = Field(default_factory=list, description='references to metrics/logs/changes')
    confidence: float = Field(..., ge=0.0, le=1.0)

class Incident(BaseModel):
    incident_id: str
    start_time: datetime
    end_time: datetime
    services: List[str]
    anomaly_type: str
    metric_summary: Dict[str, float] = Field(default_factory=dict)
    log_samples: List[str] = Field(default_factory=list)
    change_refs: List[str] = Field(default_factory=list)
    rca: Optional[RCAResult] = None
    actions: List[Action] = Field(default_factory=list)

# TelemetryRecord.schema_json(indent=2)[:400]
schema_dict = TelemetryRecord.model_json_schema()
schema_json = json.dumps(schema_dict, indent=2)
print(schema_json[:400])


{
  "properties": {
    "kind": {
      "description": "metric | log | k8s_event | change",
      "title": "Kind",
      "type": "string"
    },
    "timestamp": {
      "format": "date-time",
      "title": "Timestamp",
      "type": "string"
    },
    "service": {
      "title": "Service",
      "type": "string"
    },
    "payload": {
      "additionalProperties": true,
      "title": "Payload


## Section 7 – Quick Sanity Check

Load a few rows from the generated CSV files and instantiate example models to ensure everything is consistent.


In [14]:
# Load small samples
metrics_sample = pd.read_csv(metrics_path).head()
logs_sample = pd.read_csv(logs_path).head()
k8s_sample = pd.read_csv(k8s_path).head()
changes_sample = pd.read_csv(changes_path).head()

print('Metrics sample:')
display(metrics_sample)

print('Logs sample:')
display(logs_sample)

print('K8s events sample:')
display(k8s_sample)

print('Change events sample:')
display(changes_sample)

# Instantiate a TelemetryRecord from a log row
if not logs_sample.empty:
    row = logs_sample.iloc[0]
    rec = TelemetryRecord(
        kind='log',
        timestamp=pd.to_datetime(row['timestamp']),
        service=row['service'],
        payload={'level': row['level'], 'message': row['message']},
    )
    print('Example TelemetryRecord:', rec)


Metrics sample:


,timestamp,service,cpu_utilization,rps,latency_p95_ms,error_rate
0,2026-06-10 10:00:00,checkout-api,42.718891,169.410955,105.497667,0.010898
1,2026-06-10 10:01:00,checkout-api,40.837959,197.579256,121.601995,0.010954
2,2026-06-10 10:02:00,checkout-api,43.523207,183.704577,124.724916,0.012124
3,2026-06-10 10:03:00,checkout-api,39.961753,173.039374,113.690241,0.010688
4,2026-06-10 10:04:00,checkout-api,45.427968,188.708788,117.167827,0.010956


Logs sample:


,timestamp,service,level,message
0,2026-06-10 10:00:00,checkout-api,INFO,checkout-api handled request batch successfully
1,2026-06-10 10:01:00,checkout-api,INFO,checkout-api handled request batch successfully
2,2026-06-10 10:02:00,checkout-api,INFO,checkout-api handled request batch successfully
3,2026-06-10 10:03:00,checkout-api,INFO,checkout-api handled request batch successfully
4,2026-06-10 10:04:00,checkout-api,INFO,checkout-api handled request batch successfully


K8s events sample:


,timestamp,service,type,reason,message
0,2026-06-10 10:51:00,auth-service,Normal,CrashLoopBackOff,Pod for auth-service reported event


Change events sample:


,timestamp,service,change_type,description,version
0,2026-06-10 13:58:00,checkout-api,deploy,deploy applied to checkout-api at 2026-06-10T1...,v4.0.9
1,2026-06-10 10:40:00,auth-service,deploy,deploy applied to auth-service at 2026-06-10T1...,v3.2.2
2,2026-06-10 12:02:00,payments-service,deploy,deploy applied to payments-service at 2026-06-...,v2.5.2
3,2026-06-10 13:11:00,checkout-api,feature_flag,feature_flag applied to checkout-api at 2026-0...,v3.5.7
4,2026-06-10 11:57:00,payments-service,feature_flag,feature_flag applied to payments-service at 20...,v3.1.1


Example TelemetryRecord: kind='log' timestamp=Timestamp('2026-06-10 10:00:00') service='checkout-api' payload={'level': 'INFO', 'message': 'checkout-api handled request batch successfully'}
